## Executive Summary

- The Duka Electronics dataset contains **1,000 sales transactions** across five branches between January and June 2026.
- Nairobi recorded the highest branch revenue at **KES 26,106,100**, while Eldoret recorded the lowest at **KES 8,844,200**.
- Laptops generated the highest product revenue, contributing **KES 41,535,000** in sales.
- April was the best-performing month with revenue of **KES 13,518,300**, while June recorded the lowest revenue of **KES 10,903,400**.
- The Premium Online sales filter for Nairobi and Mombasa returned **83 transactions** worth **KES 10,218,000**.

In [23]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),

    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(
                      rng.integers(0, 180, n), unit="D"
                  ),

    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),

    "product": rng.choice(list(products), n),

    "quantity": rng.integers(1, 6, n),

    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)

In [24]:
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Dataset loaded successfully!")


Dataset loaded successfully!


In [25]:
# 1. First five records
display(sales.head())

# 2. Dataset shape
print("Dataset shape:", sales.shape)

# 3. Column data types and missing values
sales.info()

# 4. Missing values per column
print("\nMissing values:")
print(sales.isnull().sum())

# 5. Date range
print("\nDate range:")
print("Earliest date:", sales["order_date"].min())
print("Latest date:", sales["order_date"].max())

# 6. Key categories
print("\nBranches:")
print(sales["branch"].value_counts())

print("\nProducts:")
print(sales["product"].value_counts())

print("\nSales channels:")
print(sales["channel"].value_counts())

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


Dataset shape: (1000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB

Missing values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date range:
Earliest date: 2026-01-01 00:00:00
Latest date: 2026-06-29 00:00:00

Branches:
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

Products:
product
Laptop 

### First-Five-Minutes Inspection

1. **Shape:** The dataset contains 1,000 rows and 7 columns.
2. **Data types:** There are 3 integer columns, 3 object columns and 1 datetime column. The order_date column was correctly parsed as datetime64[ns].
3. **Missing values:** No missing values were detected across the seven columns.
4. **Date range:** The dataset covers 1 January 2026 to 29 June 2026, representing the first half of 2026.
5. **Key categories:** There are 5 branches, 5 products and 3 sales channels. Nairobi has the highest number of orders (376), while Eldoret has the lowest (117). In-store is the most common sales channel, accounting for 579 orders.

In [26]:
# Calculate revenue
sales["revenue"] = sales["quantity"] * sales["unit_price"]

# Extract month name
sales["month"] = sales["order_date"].dt.month_name()

# Extract weekday name
sales["weekday"] = sales["order_date"].dt.day_name()

# Identify weekend orders
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5

# Classify products by price
sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

# Display the updated dataset
display(sales.head())

print("Updated dataset shape:", sales.shape)

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


Updated dataset shape: (1000, 12)


In [27]:
# Rank branches by total revenue
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("BRANCH RANKING BY TOTAL REVENUE")
display(branch_revenue)

# Rank branches by number of orders
branch_orders = (
    sales.groupby("branch")["order_id"]
    .count()
    .sort_values(ascending=False)
)

print("BRANCH RANKING BY NUMBER OF ORDERS")
display(branch_orders)

BRANCH RANKING BY TOTAL REVENUE


,revenue
branch,
Nairobi,26106100
Mombasa,14003700
Nakuru,11970400
Kisumu,10868100
Eldoret,8844200


BRANCH RANKING BY NUMBER OF ORDERS


,order_id
branch,
Nairobi,376
Mombasa,186
Nakuru,162
Kisumu,159
Eldoret,117


### Task 3: Branch Performance

**Revenue ranking:**
1. Nairobi – KES 26,106,100
2. Mombasa – KES 14,003,700
3. Nakuru – KES 11,970,400
4. Kisumu – KES 10,868,100
5. Eldoret – KES 8,844,200

**Order ranking:**
1. Nairobi – 376 orders
2. Mombasa – 186 orders
3. Nakuru – 162 orders
4. Kisumu – 159 orders
5. Eldoret – 117 orders

**Interpretation:**

The revenue and order rankings are the same. Nairobi leads in both measures, generating KES 26,106,100 from 376 orders. Eldoret ranks last in both, with KES 8,844,200 from 117 orders. This suggests that branches with more orders also generated more total revenue in this dataset. However, order volume alone does not determine revenue because product prices and quantities also influence sales performance.

In [28]:
# Top three products by revenue
product_revenue = (
    sales.groupby("product")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("TOP THREE PRODUCTS BY REVENUE")
display(product_revenue)

# Top three products by units sold
product_quantity = (
    sales.groupby("product")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("TOP THREE PRODUCTS BY UNITS SOLD")
display(product_quantity)

TOP THREE PRODUCTS BY REVENUE


,revenue
product,
Laptop,41535000
Tablet,19348000
Phone,8880000


TOP THREE PRODUCTS BY UNITS SOLD


,quantity
product,
Tablet,691
Laptop,639
Phone,592


### Task 4: Top Three Products

**Top three products by revenue:**
1. Laptop – KES 41,535,000
2. Tablet – KES 19,348,000
3. Phone – KES 8,880,000

**Top three products by units sold:**
1. Tablet – 691 units
2. Laptop – 639 units
3. Phone – 592 units

**Interpretation:**

Laptops generate the highest revenue despite selling fewer units than Tablets. This is because laptops have a much higher unit price of KES 65,000 compared with Tablets at KES 28,000. Tablets lead in units sold, showing stronger sales volume, while Laptops contribute more financial value. Phones rank third in both measures. This demonstrates why management should consider both sales volume and revenue when evaluating product performance.

In [29]:
# Compare weekend and weekday sales
weekend_summary = (
    sales.groupby("is_weekend")
    .agg(
        number_of_orders=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        average_order_revenue=("revenue", "mean")
    )
)

# Give the groups readable names
weekend_summary.index = ["Weekday", "Weekend"]

print("WEEKEND VS WEEKDAY PERFORMANCE")
display(weekend_summary)

WEEKEND VS WEEKDAY PERFORMANCE


,number_of_orders,total_revenue,average_order_revenue
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


### Task 5: Weekend vs Weekday Performance

| Metric | Weekday | Weekend |
|---|---:|---:|
| Number of orders | 723 | 277 |
| Total revenue | KES 51,856,200 | KES 19,936,300 |
| Average order revenue | KES 71,723.65 | KES 71,972.20 |

**Interpretation:**

Weekdays account for 723 orders compared with 277 weekend orders. Consequently, weekday revenue is higher at KES 51,856,200, compared with KES 19,936,300 during weekends. However, weekend orders have a slightly higher average revenue of KES 71,972.20 compared with KES 71,723.65 on weekdays. This suggests that although fewer customers purchase during weekends, their average spending per order is marginally higher. Management could investigate weekend promotions to increase order volumes.

In [30]:
# Calculate monthly revenue
monthly_revenue = (
    sales.groupby("month")["revenue"]
    .sum()
)

# Arrange months in calendar order
month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

monthly_revenue = monthly_revenue.reindex(month_order)

print("MONTHLY REVENUE")
display(monthly_revenue)

# Identify best and worst months
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

# Calculate percentage difference relative to the worst month
percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print("Best month:", best_month)
print("Best revenue: KES", best_revenue)

print("\nWorst month:", worst_month)
print("Worst revenue: KES", worst_revenue)

print(
    "\nPercentage difference:",
    round(percentage_difference, 2), "%"
)

MONTHLY REVENUE


,revenue
month,
January,11786600
February,11772000
March,12396800
April,13518300
May,11415400
June,10903400


Best month: April
Best revenue: KES 13518300

Worst month: June
Worst revenue: KES 10903400

Percentage difference: 23.98 %


### Task 6: Monthly Revenue Performance

| Month | Total Revenue |
|---|---:|
| January | KES 11,786,600 |
| February | KES 11,772,000 |
| March | KES 12,396,800 |
| April | KES 13,518,300 |
| May | KES 11,415,400 |
| June | KES 10,903,400 |

**Best month:** April, with KES 13,518,300 in revenue.

**Worst month:** June, with KES 10,903,400 in revenue.

**Percentage difference:** 23.98%.

**Interpretation:**

April was the strongest sales month, generating KES 13,518,300, while June recorded the lowest revenue at KES 10,903,400. April's revenue was 23.98% higher than June's. This indicates a decline in sales revenue towards the end of the half-year period. Management should investigate possible reasons for the June decline and consider strategies to improve sales performance.

In [31]:
# Filter Premium online orders from Nairobi and Mombasa
premium_online_coast_capital = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Display filtered records
display(premium_online_coast_capital.head())

# Calculate number of orders and total revenue
order_count = len(premium_online_coast_capital)
total_revenue = premium_online_coast_capital["revenue"].sum()

print("Number of qualifying orders:", order_count)
print("Total revenue: KES", total_revenue)

# Export without the DataFrame index
premium_online_coast_capital.to_csv(
    "premium_online_coast_capital.csv",
    index=False
)

print("CSV exported successfully!")

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


Number of qualifying orders: 83
Total revenue: KES 10218000
CSV exported successfully!


Premium Online Orders – Nairobi and Mombasa

In [32]:
premium_online = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

display(premium_online.head())

print("Number of transactions:", len(premium_online))
print("Total revenue:", premium_online["revenue"].sum())

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


Number of transactions: 83
Total revenue: 10218000


In [33]:
import os

os.makedirs("outputs", exist_ok=True)

premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("CSV exported successfully.")

CSV exported successfully.


In [34]:
import os

print(os.path.exists(
    "outputs/premium_online_coast_capital.csv"
))

True


### Task 7: Premium Online Orders — Nairobi and Mombasa

The analysis filtered orders where the price band was Premium, the sales channel was Online, and the branch was either Nairobi or Mombasa.

**Findings:**
- Number of qualifying orders: 83
- Total revenue: KES 10,218,000

**Interpretation:**

The two branches recorded 83 qualifying Premium online orders, generating KES 10,218,000 in revenue. These results show the contribution of high-value online sales in the capital and coastal markets.

The filtered records were exported to `premium_online_coast_capital.csv` using `index=False` to avoid including the DataFrame index.

In [35]:
import os

file_path = "premium_online_coast_capital.csv"

print("File exists:", os.path.exists(file_path))

check_export = pd.read_csv(file_path)

print("Exported rows:", len(check_export))
print("Exported columns:", list(check_export.columns))
print("Exported revenue: KES", check_export["revenue"].sum())

File exists: True
Exported rows: 83
Exported columns: ['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price', 'revenue', 'month', 'weekday', 'is_weekend', 'price_band']
Exported revenue: KES 10218000


### Excel and pandas Reconciliation

The branch revenue totals calculated in Excel using SUMIFS were compared with the pandas branch revenue results. The XLOOKUP formula was used to retrieve the correct region for each branch. The PivotTable also summarised revenue by branch and sales channel. Both Excel and pandas produced the same overall revenue of **KES 71,792,500**, confirming that the spreadsheet calculations agree with the Python analysis.

In [36]:
# Verify pandas branch revenue totals

pandas_branch_totals = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_index()
)

print("PANDAS BRANCH REVENUE TOTALS:")
display(pandas_branch_totals)

print("PANDAS GRAND TOTAL:")
print(sales["revenue"].sum())

PANDAS BRANCH REVENUE TOTALS:


,revenue
branch,
Eldoret,8844200
Kisumu,10868100
Mombasa,14003700
Nairobi,26106100
Nakuru,11970400


PANDAS GRAND TOTAL:
71792500


In [37]:
from google.colab import files

files.download("duka_sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
from google.colab import files

files.download("premium_online_coast_capital.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:

import os

print(os.listdir("/content"))


['.config', 'outputs', 'premium_online_coast_capital.csv', 'duka_branch_summary.xlsx', 'duka_sales.csv', 'sample_data']


In [40]:

import os
import shutil

source = "/duka_branch_summary.xlsx"
destination = "/content/duka_branch_summary.xlsx"

if os.path.exists(source):
    shutil.move(source, destination)
    print("Workbook moved successfully!")
else:
    print("Workbook not found at the expected location.")


Workbook not found at the expected location.


In [41]:
import os

print(os.listdir("/content"))

['.config', 'outputs', 'premium_online_coast_capital.csv', 'duka_branch_summary.xlsx', 'duka_sales.csv', 'sample_data']


In [42]:

import pandas as pd

excel_path = "/content/duka_branch_summary.xlsx"

# Read the workbook
excel_file = pd.ExcelFile(excel_path)

# Display all worksheet names
print("EXCEL WORKBOOK SHEETS:")
print(excel_file.sheet_names)

# Inspect the first five rows of every worksheet
for sheet in excel_file.sheet_names:
    print(f"\n{'='*50}")
    print(f"SHEET: {sheet}")
    print("="*50)

    sheet_data = pd.read_excel(excel_path, sheet_name=sheet)
    display(sheet_data.head())
    print("Columns:", sheet_data.columns.tolist())


EXCEL WORKBOOK SHEETS:
['duka_sales', 'Branches', 'Branch Channel Pivot', 'Branch Summary']

SHEET: duka_sales


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band,region
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,NaN,NaN,NaN,NaN,Rift Valley
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,NaN,NaN,NaN,NaN,Central
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,NaN,NaN,NaN,NaN,Rift Valley
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,NaN,NaN,NaN,NaN,Central
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,NaN,NaN,NaN,NaN,Central


Columns: ['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel', 'unit_price', 'revenue', 'month', 'weekday', 'is_weekend', 'price_band', 'region']

SHEET: Branches


,branch,region
0,Nairobi,Central
1,Mombasa,Coast
2,Kisumu,Western
3,Nakuru,Rift Valley
4,Eldoret,North Rift


Columns: ['branch', 'region']

SHEET: Branch Channel Pivot


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,Sum of revenue,Column Labels,NaN,NaN,NaN
2,Row Labels,In-store,Online,Phone order,Grand Total
3,Eldoret,4885300,3450700,508200,8844200
4,Kisumu,6468600,3370800,1028700,10868100


Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

SHEET: Branch Summary


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


,Branch,total_revenue
0,Nairobi,26106100
1,Mombasa,14003700
2,Kisumu,10868100
3,Nakuru,11970400
4,Eldoret,8844200


Columns: ['Branch', 'total_revenue']


In [43]:

import pandas as pd
import numpy as np

excel_path = "/content/duka_branch_summary.xlsx"

# 1. Compare Excel SUMIFS with pandas branch revenue
excel_summary = pd.read_excel(
    excel_path,
    sheet_name="Branch Summary"
)

pandas_summary = (
    sales.groupby("branch")["revenue"]
    .sum()
    .reset_index()
)

comparison = pandas_summary.merge(
    excel_summary,
    left_on="branch",
    right_on="Branch",
    how="inner"
)

comparison["difference"] = (
    comparison["revenue"] - comparison["total_revenue"]
)

comparison["SUMIFS_match"] = np.isclose(
    comparison["revenue"],
    comparison["total_revenue"]
)

print("1. EXCEL SUMIFS VS PANDAS BRANCH REVENUE")
display(comparison)

# 2. Verify Excel XLOOKUP region results
excel_sales = pd.read_excel(
    excel_path,
    sheet_name="duka_sales"
)

branch_mapping = pd.read_excel(
    excel_path,
    sheet_name="Branches"
)

region_lookup = branch_mapping.set_index("branch")["region"]

expected_regions = excel_sales["branch"].map(region_lookup)

xlookup_match = expected_regions.equals(excel_sales["region"])

print("\n2. EXCEL XLOOKUP REGION VERIFICATION")
print("All Excel region results match the branch mapping:", xlookup_match)

# 3. Compare Excel PivotTable with pandas pivot table
pivot_raw = pd.read_excel(
    excel_path,
    sheet_name="Branch Channel Pivot",
    header=None
)

header_row = pivot_raw.index[
    pivot_raw.iloc[:, 0].astype(str).str.strip() == "Row Labels"
][0]

pivot_headers = pivot_raw.iloc[header_row].tolist()

excel_pivot = pivot_raw.iloc[header_row + 1:].copy()
excel_pivot.columns = pivot_headers

excel_pivot = excel_pivot.dropna(subset=["Row Labels"])
excel_pivot = excel_pivot[
    excel_pivot["Row Labels"] != "Grand Total"
].set_index("Row Labels")

pandas_pivot = pd.pivot_table(
    sales,
    index="branch",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0
)

pandas_pivot["Grand Total"] = pandas_pivot.sum(axis=1)

excel_pivot = excel_pivot.apply(pd.to_numeric)

pandas_pivot = pandas_pivot.reindex(
    index=excel_pivot.index,
    columns=excel_pivot.columns
)

pivot_match = np.allclose(
    excel_pivot.to_numpy(),
    pandas_pivot.to_numpy()
)

print("\n3. EXCEL PIVOTTABLE VS PANDAS PIVOT")
print("All branch and channel totals match:", pivot_match)

# 4. Overall reconciliation results
print("\n4. RECONCILIATION SUMMARY")
print("SUMIFS branch totals match:", comparison["SUMIFS_match"].all())
print("XLOOKUP region results match:", xlookup_match)
print("PivotTable totals match:", pivot_match)

print("\nPandas grand total: KES", sales["revenue"].sum())
print("Excel grand total: KES", excel_summary["total_revenue"].sum())


1. EXCEL SUMIFS VS PANDAS BRANCH REVENUE


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


,branch,revenue,Branch,total_revenue,difference,SUMIFS_match
0,Eldoret,8844200,Eldoret,8844200,0,True
1,Kisumu,10868100,Kisumu,10868100,0,True
2,Mombasa,14003700,Mombasa,14003700,0,True
3,Nairobi,26106100,Nairobi,26106100,0,True
4,Nakuru,11970400,Nakuru,11970400,0,True



2. EXCEL XLOOKUP REGION VERIFICATION
All Excel region results match the branch mapping: True

3. EXCEL PIVOTTABLE VS PANDAS PIVOT
All branch and channel totals match: True

4. RECONCILIATION SUMMARY
SUMIFS branch totals match: True
XLOOKUP region results match: True
PivotTable totals match: True

Pandas grand total: KES 71792500
Excel grand total: KES 71792500
